### In this notebook, we will go deeper in the shots analysis of Messi.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from statsbombpy import sb

warnings.filterwarnings("ignore")

### Now let's meake a loop to being more flexible and analyse different matches easily 

In [2]:
COMPETITION_ID = 11   # La Liga
SEASON_ID = 23        # 2011/2012

matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
match_ids = matches["match_id"].tolist()

infos_match = matches[["match_id", "match_date", "home_team", "away_team",
                       "home_score", "away_score"]]
print(len(match_ids), "matchs")

37 matchs


In [3]:
tous_les_tirs = []

for i, mid in enumerate(match_ids):
    ev = sb.events(match_id=mid)
    tirs = ev[(ev["type"] == "Shot") & (ev["player"].str.contains("Messi", na=False))]
    tous_les_tirs.append(tirs)
    print(f"Match {i+1}/{len(match_ids)} : {len(tirs)} tirs")

messi_shots = pd.concat(tous_les_tirs, ignore_index=True)
print(messi_shots.shape)

Match 1/37 : 8 tirs
Match 2/37 : 10 tirs
Match 3/37 : 7 tirs
Match 4/37 : 7 tirs
Match 5/37 : 6 tirs
Match 6/37 : 1 tirs
Match 7/37 : 8 tirs
Match 8/37 : 7 tirs
Match 9/37 : 3 tirs
Match 10/37 : 7 tirs
Match 11/37 : 4 tirs
Match 12/37 : 7 tirs
Match 13/37 : 9 tirs
Match 14/37 : 1 tirs
Match 15/37 : 6 tirs
Match 16/37 : 7 tirs
Match 17/37 : 2 tirs
Match 18/37 : 6 tirs
Match 19/37 : 4 tirs
Match 20/37 : 5 tirs
Match 21/37 : 6 tirs
Match 22/37 : 5 tirs
Match 23/37 : 3 tirs
Match 24/37 : 2 tirs
Match 25/37 : 5 tirs
Match 26/37 : 6 tirs
Match 27/37 : 4 tirs
Match 28/37 : 3 tirs
Match 29/37 : 5 tirs
Match 30/37 : 6 tirs
Match 31/37 : 1 tirs
Match 32/37 : 8 tirs
Match 33/37 : 4 tirs
Match 34/37 : 5 tirs
Match 35/37 : 7 tirs
Match 36/37 : 7 tirs
Match 37/37 : 6 tirs
(198, 116)


### this loop help us to identify the number of shots of Messi during these different games. Now let's add somme context. 

In [4]:
saison = messi_shots.merge(infos_match, on="match_id", how="left")

saison["adversaire"] = np.where(saison["home_team"] == "Barcelona",
                                saison["away_team"], saison["home_team"])
saison["domicile"] = saison["home_team"] == "Barcelona"

### this cell help us to understand the game (home team, opponents...)

In [5]:
print("Tirs     :", len(saison))
print("Buts     :", (saison["shot_outcome"] == "Goal").sum())
print("xG total :", round(saison["shot_statsbomb_xg"].sum(), 1))
print("Penaltys :", (saison["shot_type"] == "Penalty").sum())

Tirs     : 198
Buts     : 50
xG total : 35.7
Penaltys : 11


### here we have the stats of his season (incredible...). let's save in a CSV to continue easily after. 

In [6]:
saison["x"] = saison["location"].str[0]
saison["y"] = saison["location"].str[1]

colonnes = ["match_date", "adversaire", "domicile", "minute", "x", "y",
            "shot_statsbomb_xg", "shot_outcome", "shot_body_part", "shot_type"]

os.makedirs("../data", exist_ok=True)
saison[colonnes].to_csv("../data/messi_tirs_2011_12.csv", index=False)